# C12 · Crosswinds at Santa Barbara: when can a student fly solo?

| | |
|---|---|
| **Difficulty** | ★★★★☆ |
| **Time** | 4-5 hours |
| **Data** | Hourly METAR wind reports from Santa Barbara Municipal Airport (ASOS station KSBA), 2021-2023, 26,101 reports (Iowa Environmental Mesonet archive) |
| **Skills** | Vectors from wind reports (crosswind components, true vs magnetic headings) · calms as censored speeds · discrete (rounded) circular data · direction models that depend on speed, hour and season · log-Fourier (generalised von Mises) densities · posterior predictive checks of a tail probability · held-out-year calibration · clustered (autocorrelated) events and a day-level random effect · turning a posterior into a schedule |

## The brief

A flight school at Santa Barbara (SBA) sends student pilots on **solo** flights. Its safety
rules for a student solo, as far as wind goes:

> "The **crosswind component** on the runway in use must not exceed **8 knots**. Calm is fine.
> Students only use runway **7/25** (the long runway); the tower gives them the end that faces
> into the wind, so tailwind is not an issue. First-solo students have a stricter **6-knot**
> limit."

The chief instructor asks:

> "By month and hour of the day, what is the probability that a solo slot is flyable? We want a
> **default two-hour solo window** between 08:00 and 18:00 local time. Which one should it be,
> how sure are you, and how many bookings a month should we expect to cancel because of the
> wind? Does the answer change for first-solo students?"

You have three years of hourly wind reports from the airport's own weather station. The
obvious approach - fit a model for the wind speed, fit a model for the wind direction, and
combine them - gives a confident, plausible, wrong answer. Find out why, fix it, and then check
your probabilities on a year the model has not seen.

This is the applied sequel to example **E81** (circular data). Do E81 first: this challenge
assumes you know the mean resultant vector, the canonical (natural-parameter) von Mises link,
circular predictors as harmonics, and what the Santa Barbara wind does over a day. E81 modelled
the *direction*; here the question is about a **vector** - speed *and* direction together - and
about a small tail probability.

## How this notebook works

- Each task states **what to deliver**, not how. Write your code in the `YOUR CODE HERE` cells.
- Stuck? `h.hint("task2")` reveals hints one level at a time: *nudge → approach → code skeleton*.
  Try to get by on nudges.
- `h.check("task2", ...)` compares your numbers with the reference solution.
- A full worked solution lives in `notebooks/solutions/`. Open it only when you are done (or truly stuck).

In [ ]:
import logging
import time

import arviz as az
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
from scipy import special, stats

from pymc_challenges import Hints, data

RANDOM_SEED = 1932
rng = np.random.default_rng(RANDOM_SEED)
az.style.use("arviz-variat")
logging.getLogger("pymc").setLevel(logging.WARNING)  # several fits: no sampler banner per fit
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
T_START = time.time()

h = Hints("C12")
h.tasks()

## The data

Routine hourly reports (METARs), issued at minute 53 of each hour. `drct` is the direction the
wind blows **from**, in degrees clockwise from **true** north, reported in steps of 10; `sknt` is
the speed in knots. A calm is reported as direction 0 and speed 0. An empty direction with a
non-zero speed is a *variable* wind (no steady direction).

The school books slots in **local clock time**, so we convert the UTC timestamps to
`America/Los_Angeles` (daylight-saving time included). We fit on **2021-2022** and keep
**2023** aside: nothing you do before Task 4 may look at it.

In [ ]:
data.describe("ksba_wind")
raw = data.load("ksba_wind")
local = pd.to_datetime(raw.valid).dt.tz_localize("UTC").dt.tz_convert("America/Los_Angeles")
wind = raw.assign(hour=local.dt.hour, month=local.dt.month, year=local.dt.year, date=local.dt.date)
wind = wind[(wind.year >= 2021) & wind.sknt.notna()].reset_index(drop=True)   # 8 reports fall on 31 Dec 2020 local
train = wind[wind.year < 2023].reset_index(drop=True)
test = wind[wind.year == 2023].reset_index(drop=True)
print(f"training reports (2021-2022): {len(train):,}; held-out 2023: {len(test):,}")
train.head()

Runway 7/25, from the FAA airport record (form 5010, as listed e.g. on AirNav): 6,052 ft long,
**magnetic** headings 075° / 255°, **true** headings 089° / 269°. (Runways are named after their
magnetic heading rounded to 10°; the local magnetic variation is about 12-14° east.)

In [ ]:
RWY_MAGNETIC = 75.0
RWY_TRUE = 89.0
LIMIT = 8.0          # knots of crosswind, solo students
LIMIT_FIRST = 6.0    # knots of crosswind, first-solo students
DAY_HOURS = range(8, 18)   # the school is open 08:00-18:00

## Task 0 · Look at the data

**Deliver**
1. How many reports are calm, how many have no direction, and how fast is the wind when the
   direction is missing? What are the possible values of `drct` and `sknt`?
2. Rose diagrams of the **afternoon** wind (12:00-16:00), split by speed class (light, moderate,
   strong). Does the direction depend on the speed?

In [ ]:
# YOUR CODE HERE

In [ ]:
# h.hint("task0")

## Task 1 · The crosswind, and what the raw data can say

**Deliver**
1. For every report, the crosswind component on runway 7/25 and whether the slot is **flyable**
   under the 8-knot rule. Decide what calms and missing directions mean. Why does the choice of
   runway end not matter for the crosswind?
2. The share of 2021-2022 reports that exceed the limit, with the **true** heading - and with the
   magnetic heading 075° used by mistake. Which error does the mistake make, and how big is it?
3. A month x hour table of the observed exceedance rate. Is it good enough to schedule from?

In [ ]:
# YOUR CODE HERE

In [ ]:
# h.hint("task1")
# h.check("task1", exceed_true=..., exceed_magnetic=...)   # shares of ALL 2021-2022 reports above 8 kt

## Task 2 · The obvious model: speed and direction separately

A crosswind exceedance needs a strong wind *and* the wrong direction. The natural plan is to
model each and combine them.

**Deliver**
1. A model for the wind **speed** as a smooth function of hour and month that treats the calms
   honestly (they are speeds below a threshold, not a separate kind of wind) and respects that
   speeds are reported in whole knots. Check it on the share of calms and of strong winds by hour.
2. A model for the reported **direction** as a smooth function of hour and month. Make it flexible
   enough to have more than one preferred direction (E81 showed that one von Mises is not).
3. Combine them into the probability of exceeding 8 knots of crosswind for every month x hour,
   and compare it, hour by hour, with the observed rate. What goes wrong, and why?

In [ ]:
# YOUR CODE HERE

In [ ]:
# h.hint("task2")
# h.check("task2", calm_0700=..., exceed_afternoon_indep=...)
#   calm_0700: predicted share of calm reports at 07:53, averaged over months
#   exceed_afternoon_indep: predicted exceedance rate averaged over 11:00-15:59 and all months

## Task 3 · Let the direction depend on the speed

**Deliver**
1. A direction model that conditions on the reported speed as well as on hour and month. Try the
   canonical von Mises of E81 first, then a more flexible circular family. Combined with your speed
   model, this is a model of the full wind vector.
2. The exceedance check of Task 2, by hour **and** by month, for both. Which one can you use?
3. Show *where* the inadequate one goes wrong: compare the predicted and observed directions of
   strong afternoon winds.

In [ ]:
# YOUR CODE HERE

In [ ]:
# h.hint("task3")
# h.check("task3", exceed_afternoon_vm=..., exceed_afternoon_flex=...)   # same average as in Task 2

## Task 4 · Is the model honest about 2023?

The posterior bands in Task 3 are narrow. But the model treats 17,000 hourly reports as
independent pieces of evidence, and wind is not like that.

**Deliver**
1. For each month of **2023**, the observed share of daytime (08:00-17:59) reports above the limit,
   and your model's 90% predictive interval for it. How many of the 12 months fall inside?
2. Evidence from 2021-2022 that bad hours are **clustered** (compare what the model expects with
   what happened at the level of days).
3. A model extension that represents the clustering, refitted on 2021-2022 only, and the same
   2023 check.

In [ ]:
# YOUR CODE HERE

In [ ]:
# h.hint("task4")
# h.check("task4", covered_iid=..., covered_day=..., sigma_day=...)
#   covered_*: number of 2023 months (of 12) inside the 90% predictive interval; sigma_day: posterior mean

## Task 5 · The schedule

A solo slot is a two-hour window $[h, h+2)$; call it flyable if both reports made inside it (at
$h$:53 and $h+1$:53) are within the limit.

**Deliver**
1. For every month and every window starting between 08:00 and 16:00, the probability that the
   window is flyable under the 8-knot rule, from your final model (with its uncertainty).
2. The recommended **default window** for the whole year: its average probability, the posterior
   probability that it is the best window, and how much better it is than the runner-up and the
   worst. Would a month-specific window be worth the complication?
3. For a student with 20 solo bookings in March in the default window, the expected number
   cancelled because of the wind, with a 90% predictive interval.
4. The same window ranking for **first-solo students** (6-knot limit). Does the advice change?

In [ ]:
# YOUR CODE HERE

In [ ]:
# h.hint("task5")
# h.check("task5", p_default=..., p_best_default=..., cancel_march=..., p_default_first=...)
#   p_default: year-average P(flyable) of the recommended 8-kt window; p_best_default: P(it is the best);
#   cancel_march: expected cancellations of 20 March bookings; p_default_first: same window, 6-kt limit

## Going further

- **Gusts.** The METAR also reports gusts (not in this extract). Download them from the IEM archive,
  add a gust model conditional on the mean speed and direction, and redo Task 5 with a gust
  crosswind limit. How much of the morning advantage survives?
- **Persistence.** Replace the day effect with a latent AR(1) process over hours (or an HMM over
  weather regimes) and check whether "the next two hours" can be *forecast* from the last report -
  a much more useful product for a dispatcher than climatology.
- **Other runway.** Runway 15/33 (true headings 166/346) is shorter but nearly across 7/25. Under
  a rule "either runway", how much flyable time does the school gain, and which wind regimes does
  the second runway rescue?
- **Continuous direction.** Replace the 36-bin categorical with a continuous log-Fourier density
  plus a rounding model (E81 handled rounding by jittering). Do the tail probabilities change?

In [ ]:
h.progress()